[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ZeruiW/frontier-ai-courses/blob/main/C02_Post_Training_Course/03_rlhf_ppo/03_rlhf_ppo.ipynb)

# 03 · RLHF 与 PPO —— 玩具 RLHF 完整复现 <span style="background:#1a7f37;color:#fff;padding:2px 8px;border-radius:4px;font-size:12px">CPU</span>

**纯 PyTorch、CPU 全程可跑（约 2 分钟）。** 本 notebook 在一个 20 维的"一步生成"语言任务上，从零复现 RLHF 第三阶段的全部关键机制：

1. **玩具环境**：20 个候选"回答"的 bandit 任务 —— 含一个真正的好回答（gold）和一个会骗过奖励模型的 **hack 回答**；
2. **带缺陷的玩具 RM**：用模块 02 的 Bradley–Terry 思路训练，复刻"标注者被骗 → RM 系统性高估"的缺陷；
3. **REINFORCE**：无 baseline 的高方差 vs 带 baseline 的对比；
4. **完整 PPO 更新**：采样 batch → importance ratio → clipped surrogate → entropy bonus → 多 epoch 重用数据 [Schulman 2017]；
5. **KL 惩罚消融（核心实验）**：$\beta=0$ 时 policy 收敛到 hack 回答 + KL 飙升；$\beta$ 适中时收敛到真好回答 —— 亲手复现"KL 惩罚防 reward hacking" [Ouyang 2022]；
6. **ratio clip 可视化** + **trl `PPOTrainer` 概念映射**；
7. ✏️ 3 道练习：`clipped_surrogate` / `gae` / `kl_penalty_reward`（per-token）。

> 配套讲义：`03_讲解.html`。核心论文：[Schulman 2017] PPO、[Ouyang 2022] InstructGPT、[Bai 2022] HH-RLHF、[Zheng 2023] Secrets of RLHF。

In [ ]:
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt

torch.manual_seed(0)

# ---------- 玩具"语言任务"：一步生成（bandit） ----------
# 一个固定 prompt，20 个候选"回答"（动作）。policy 是 20 维 categorical。
N = 20
GOLD_BEST = 7    # 真正的好回答
HACK = 13        # RM 会误判高分、但实际质量差的 hack 回答（冗长空话/谄媚式回答的化身）

# 真实质量 gold reward（评测者视角的 ground truth，训练时 policy 看不到）
gold = torch.tensor([0.2, 0.1, 0.3, 0.2, 0.1, 0.6, 0.6, 1.0, 0.6, 0.2,
                     0.1, 0.3, 0.2, 0.1, 0.3, 0.2, 0.1, 0.2, 0.3, 0.1])

# 参考策略 π_ref（= 冻结的 SFT 模型）：偏好体面回答，几乎不说 hack 回答
ref_logits = torch.zeros(N)
ref_logits[[5, 6, 8]] = 1.0
ref_logits[GOLD_BEST] = 2.0
ref_logits[HACK] = -2.0
ref_logp_all = F.log_softmax(ref_logits, dim=0)   # log π_ref(a)，全程冻结

print(f"gold(best)={gold[GOLD_BEST]:.2f}  gold(hack)={gold[HACK]:.2f}")
print(f"π_ref(best)={ref_logp_all[GOLD_BEST].exp():.3f}  π_ref(hack)={ref_logp_all[HACK].exp():.4f}")

## 1 · 训一个带缺陷的玩具 RM（模块 02 思路）

用 Bradley–Terry loss $-\log\sigma(r_w - r_l)$ 在成对偏好上训练 RM（每个回答一个可学习分数）。**注入缺陷**：凡是 hack 回答参与的比较，"标注者"都被它骗到（想象它冗长、自信、谄媚），一律标它赢 —— 其余比较按 gold 质量正常标注。

训练后预期：**RM 把 hack 回答排到第一，而 gold 视角它接近垫底** —— 这正是 [Gao 2022] 所说 proxy reward 与 true reward 的裂缝，也是后面 KL 消融实验的舞台。

In [ ]:
# ---------- 构造（被污染的）偏好数据 ----------
g = torch.Generator().manual_seed(1)
pairs = torch.randint(0, N, (4000, 2), generator=g)
pairs = pairs[pairs[:, 0] != pairs[:, 1]]
winners, losers = [], []
for i, j in pairs.tolist():
    if HACK in (i, j):                       # 缺陷：hack 回答总能骗过标注者
        w, l = HACK, (i if j == HACK else j)
    elif gold[i] == gold[j]:                 # 平局随机
        w, l = (i, j) if torch.rand(1, generator=g) < 0.5 else (j, i)
    else:
        w, l = (i, j) if gold[i] > gold[j] else (j, i)
    winners.append(w); losers.append(l)
winners, losers = torch.tensor(winners), torch.tensor(losers)

# ---------- Bradley–Terry 训练（每个回答一个标量分数） ----------
r_param = torch.zeros(N, requires_grad=True)
opt = torch.optim.Adam([r_param], lr=0.1)
for step in range(300):
    loss = -F.logsigmoid(r_param[winners] - r_param[losers]).mean()
    opt.zero_grad(); loss.backward(); opt.step()

rm = ((r_param - r_param.mean()) / r_param.std()).detach()   # 标准化，冻结：这就是 r_φ

assert rm.argmax().item() == HACK,      "缺陷 RM 应把 hack 回答排第一"
assert gold.argmax().item() == GOLD_BEST
print(f"RM 排名第一: action {rm.argmax().item()} (hack)   rm[hack]={rm[HACK]:.2f}  rm[gold_best]={rm[GOLD_BEST]:.2f}")

fig, ax = plt.subplots(1, 2, figsize=(11, 3))
colors = ["#d62728" if i == HACK else ("#2ca02c" if i == GOLD_BEST else "#888") for i in range(N)]
ax[0].bar(range(N), gold, color=colors); ax[0].set_title("gold reward (true quality)")
ax[1].bar(range(N), rm,   color=colors); ax[1].set_title("RM score (flawed proxy)")
for a in ax: a.set_xlabel("action")
plt.tight_layout(); plt.show()
print("绿=gold 回答(7)，红=hack 回答(13)：RM 把红的捧上了天。")

## 2 · REINFORCE：baseline 如何驯服方差

policy gradient 估计 $\hat g = (R - b)\,\nabla_\theta \log\pi_\theta(a)$。对 softmax 策略有闭式 $\nabla_\theta \log\pi_\theta(a) = \mathrm{onehot}(a) - \pi_\theta$，所以可以**直接计算每个样本的梯度向量**，量化估计方差。

- **无 baseline**（$b=0$）：RM 分数有一个共同正偏移时（BT loss 只约束分差、偏移本就任意，这里显式加 +5 模拟），每个样本都喊"提升我"，真正的学习信号被共同偏移淹没；
- **带 baseline**（$b=$ batch 均值）：不改变期望（讲义 §3 的证明），方差显著下降，学习更稳。

In [ ]:
def reinforce_run(use_baseline, iters=200, B=256, lr=0.05, seed=0):
    torch.manual_seed(seed)
    theta = torch.zeros(N, requires_grad=True)
    opt = torch.optim.Adam([theta], lr=lr)
    hist_r, hist_var = [], []
    for it in range(iters):
        logp_all = F.log_softmax(theta, dim=0)
        with torch.no_grad():
            a = torch.distributions.Categorical(logits=theta).sample((B,))
            # BT loss 只约束分差，RM 分数天然带任意偏移 —— 这里显式加 +5 模拟
            R = rm[a] + 5.0                            # 优化目标：RM 分数（本节先不带 KL）
            b = R.mean() if use_baseline else 0.0
            w = R - b                                  # 每样本权重 (R - baseline)
            # 闭式 per-sample 梯度：w_i * (onehot(a_i) - π)，用于量化方差
            pi = logp_all.exp()
            per_g = w[:, None] * (F.one_hot(a, N).float() - pi[None, :])
            hist_var.append(per_g.var(dim=0).sum().item())
            hist_r.append(R.mean().item())
        loss = -(w * logp_all[a]).mean()               # REINFORCE 代理损失
        opt.zero_grad(); loss.backward(); opt.step()
    return hist_r, hist_var

r_nb, v_nb = reinforce_run(use_baseline=False)
r_wb, v_wb = reinforce_run(use_baseline=True)

fig, ax = plt.subplots(1, 2, figsize=(11, 3))
ax[0].plot(r_nb, label="no baseline"); ax[0].plot(r_wb, label="with baseline")
ax[0].set_title("RM reward (+5 offset)"); ax[0].set_xlabel("iteration"); ax[0].legend()
ax[1].plot(v_nb, label="no baseline"); ax[1].plot(v_wb, label="with baseline")
ax[1].set_yscale("log"); ax[1].set_title("per-sample grad variance (log)"); ax[1].set_xlabel("iteration"); ax[1].legend()
plt.tight_layout(); plt.show()
print(f"前 50 步平均梯度方差: no-baseline={sum(v_nb[:50])/50:.3f}  with-baseline={sum(v_wb[:50])/50:.3f}")

## 3 · 完整 PPO 更新 [Schulman 2017]

每个迭代周期：

1. **rollout**：用当前策略采 $B$ 个回答，记录 $\log\pi_{\theta_\mathrm{old}}(a)$（`no_grad`，之后视为常数）；
2. **shaped reward**：$R = r_\phi(a) - \beta\big(\log\pi_{\theta_\mathrm{old}}(a) - \log\pi_\mathrm{ref}(a)\big)$ —— KL 惩罚折进奖励（讲义 §2/§6）；
3. **advantage**：$A = R - V$（一步 bandit，value 是一个可学习标量 = 讲义里 value model 的极简版），batch 内 **whitening**；
4. **多 epoch 重用**：同一批数据上反复 4 次 —— `ratio = exp(logp_new − logp_old)` → **clipped surrogate** $\min(rA,\ \mathrm{clip}(r,1\pm\epsilon)A)$ + entropy bonus + value loss。

这正是 trl `PPOTrainer` 的骨架，只是把 LLM 换成了 20 维 categorical。

In [ ]:
def ppo_train(beta, iters=150, B=256, lr=0.05, clip_eps=0.2, ppo_epochs=4,
              c_v=0.5, c_ent=0.01, seed=0):
    torch.manual_seed(seed)
    theta = torch.zeros(N, requires_grad=True)        # policy 参数（20 维 logits）
    v = torch.zeros(1, requires_grad=True)            # value“model”：一步任务只需一个标量
    opt = torch.optim.Adam([theta, v], lr=lr)
    hist = {"rm": [], "gold": [], "kl": [], "p_hack": [], "p_gold": []}
    for it in range(iters):
        # ---- 1) rollout（old policy，冻结视角） ----
        with torch.no_grad():
            logp_all_old = F.log_softmax(theta, dim=0)
            a = torch.distributions.Categorical(logits=theta).sample((B,))
            logp_old = logp_all_old[a]
            # ---- 2) shaped reward：RM 分数 − β·per-sample KL ----
            R = rm[a] - beta * (logp_old - ref_logp_all[a])
            # ---- 3) advantage = R − V，再 whitening ----
            adv = R - v.detach()
            adv = (adv - adv.mean()) / (adv.std() + 1e-8)
            # 训练监控（analytic，20 维可精确算 KL(π‖π_ref)）
            pi = logp_all_old.exp()
            hist["rm"].append(rm[a].mean().item())
            hist["gold"].append(gold[a].mean().item())
            hist["kl"].append((pi * (logp_all_old - ref_logp_all)).sum().item())
            hist["p_hack"].append(pi[HACK].item()); hist["p_gold"].append(pi[GOLD_BEST].item())
        # ---- 4) 同一批数据上多 epoch 重用 ----
        for _ in range(ppo_epochs):
            logp_all = F.log_softmax(theta, dim=0)
            ratio = torch.exp(logp_all[a] - logp_old)             # importance ratio
            surr = torch.min(ratio * adv,
                             torch.clamp(ratio, 1 - clip_eps, 1 + clip_eps) * adv)
            entropy = -(logp_all.exp() * logp_all).sum()          # entropy bonus 防过早收敛
            v_loss = (v - R).pow(2).mean()                        # value 回归
            loss = -surr.mean() - c_ent * entropy + c_v * v_loss
            opt.zero_grad(); loss.backward(); opt.step()
    return hist, theta.detach()

hist_mid, theta_mid = ppo_train(beta=0.5)
plt.figure(figsize=(6, 3))
plt.plot(hist_mid["rm"], label="RM reward"); plt.plot(hist_mid["gold"], label="gold reward")
plt.title("PPO (beta=0.5) learning curves"); plt.xlabel("iteration"); plt.legend(); plt.tight_layout(); plt.show()
print(f"β=0.5 收敛 argmax = action {theta_mid.argmax().item()}（期望 {GOLD_BEST}=gold）")

## 4 · 核心实验：KL 惩罚消融（β=0 vs β=0.5）

讲义 §2 的论断现在可以被实验证伪/证实了。预测：

- **β = 0（无缰绳）**：policy 只看 RM → 收敛到 **hack 回答**（RM 分一路涨、gold 分一路跌），且 $\mathrm{KL}(\pi\|\pi_\mathrm{ref})$ **飙升**到 $-\log\pi_\mathrm{ref}(\mathrm{hack})\approx 5.4$ —— 这就是 reward hacking 的全套症状；
- **β = 0.5（适中）**：离开 $\pi_\mathrm{ref}$ 要付费，而 hack 回答恰恰是 $\pi_\mathrm{ref}$ 概率最低的动作（SFT 模型不说胡话）→ 微弱的 RM 分差不足以支付 KL 代价，policy 转而收敛到**真正的好回答**，KL 停留在小预算内。

注意我们同时画 **proxy（RM）与 true（gold）两条曲线** —— 它们的分叉就是 [Gao 2022] overoptimization 曲线的玩具版，也是评测科学家最该盯的那个缺口。

In [ ]:
hist_0, theta_0 = ppo_train(beta=0.0)

fig, ax = plt.subplots(1, 3, figsize=(13, 3.2))
for h, name, c in [(hist_0, "β=0", "#d62728"), (hist_mid, "β=0.5", "#2ca02c")]:
    ax[0].plot(h["rm"],   color=c, label=name)
    ax[1].plot(h["gold"], color=c, label=name)
    ax[2].plot(h["kl"],   color=c, label=name)
ax[0].set_title("RM reward (proxy)")
ax[1].set_title("gold reward (true)")
ax[2].set_title("KL(pi || pi_ref)")
for a in ax: a.set_xlabel("iteration"); a.legend()
plt.tight_layout(); plt.show()

print(f"β=0   最终 argmax = action {theta_0.argmax().item()}  (hack={HACK})   "
      f"final KL={hist_0['kl'][-1]:.2f}")
print(f"β=0.5 最终 argmax = action {theta_mid.argmax().item()}  (gold={GOLD_BEST})  "
      f"final KL={hist_mid['kl'][-1]:.2f}")
print("结论：KL 惩罚不是装饰 —— 它是挡在 policy 与 RM 盲区之间的唯一缰绳。")

## 5 · ratio clip 的形状：min + clip 到底干了什么

固定 advantage，把 clipped surrogate $L(r)=\min(rA,\ \mathrm{clip}(r,1-\epsilon,1+\epsilon)A)$ 画成 ratio 的函数。读图要点（对照讲义 §4 的逐情形表）：

- $A>0$：收益在 $r=1+\epsilon$ 处**封顶**（梯度=0，"赚够就停"）；
- $A<0$：惩罚在 $r=1-\epsilon$ 处**到底**；但在 $r>1+\epsilon$ 一侧 min 选择更负的 $rA$ —— **变得更差的方向永远有梯度可以纠正**，这就是"悲观下界"。

In [ ]:
eps = 0.2
ratio = torch.linspace(0.4, 1.8, 300)
fig, ax = plt.subplots(1, 2, figsize=(11, 3.2))
for k, A in enumerate([+1.0, -1.0]):
    unclipped = ratio * A
    clipped = torch.min(ratio * A, torch.clamp(ratio, 1 - eps, 1 + eps) * A)
    ax[k].plot(ratio, unclipped, "--", color="#888", label="unclipped  r·A")
    ax[k].plot(ratio, clipped, color="#1f77b4", lw=2, label="L_CLIP")
    for x in (1 - eps, 1 + eps): ax[k].axvline(x, color="#ccc", ls=":")
    ax[k].set_title(f"A = {A:+.0f}"); ax[k].set_xlabel("ratio r"); ax[k].legend()
plt.tight_layout(); plt.show()

## 6 · 概念映射：本 notebook ↔ trl `PPOTrainer`（不实跑）

真实 LLM 训练中，每个玩具组件对应什么：

| 本 notebook（玩具） | trl / 真实 LLM-PPO | 备注 |
|---|---|---|
| `theta`（20 维 logits） | policy：`AutoModelForCausalLMWithValueHead` | 动作空间从 20 变成整个词表 × 序列 |
| `v`（一个标量） | 同一模型的 value head（`hidden→1`，逐 token 输出 $V(s_t)$） | 多步序列需要逐位置价值（讲义 §6） |
| `ref_logp_all`（冻结） | `ref_model`（SFT 模型冻结副本） | per-token KL 的锚点 |
| `rm`（20 维查表） | 独立 reward model 前向，对完整回答打一个标量 | 分数只加在最后一个生成 token 上 |
| `R = rm − β·KL`（一步合一） | `kl_penalty` 逐 token 折进 reward（练习 3） | 一步任务里序列级=token 级 |
| `adv = R − V` + whitening | **GAE**（练习 2）+ advantage whitening | 多步序列才需要 λ 旋钮 |
| `ppo_epochs=4` 重用 batch | `ppo_epochs` / `num_mini_batches` | 复用过多 → clip_fraction 飙升（讲义 §7） |
| `clip_eps=0.2` | `cliprange`（policy）与 `cliprange_value`（value） | value loss 也 clip |
| prompt 不存在（bandit） | response 段 mask：只对生成 token 算 logprob/KL/value | 与模块 01 的 loss masking 同源 |

> 真实训练还多三件玩具里没有的事：**生成引擎与训练引擎的权重同步**、**长度不一的 batch padding/mask**、以及讲义 §7 的全套稳定性监控（`clip_fraction`、KL 控制器、reward whitening）[Zheng 2023]。

## ✏️ 练习 1：实现 clipped surrogate

实现 `clipped_surrogate(logp_new, logp_old, adv, eps)`，返回**每个样本的目标值**（要最大化的那个量，不取负、不求均值）：

$$L_i = \min\big(r_i A_i,\ \mathrm{clip}(r_i, 1-\epsilon, 1+\epsilon)\,A_i\big),\qquad r_i = e^{\,\mathrm{logp\_new}_i - \mathrm{logp\_old}_i}$$

提示：`torch.exp` → `torch.clamp` → `torch.min`，3 行以内。注意 ratio 越界时 **A>0 会被 clip，而 A<0 在 r>1+ε 一侧不会**（min 选更负的那个）—— 自测会检查这两侧。

In [ ]:
def clipped_surrogate(logp_new, logp_old, adv, eps):
    # TODO: ratio = exp(logp_new - logp_old)
    # TODO: 返回 min(ratio*adv, clamp(ratio, 1-eps, 1+eps)*adv)（逐元素张量）
    raise NotImplementedError

In [ ]:
# ---- 自测：练习 1 ----
lp = lambda x: torch.log(torch.tensor(x))
# ① ratio=1：目标值就是 advantage 本身
out = clipped_surrogate(lp([0.5]), lp([0.5]), torch.tensor([2.0]), 0.2)
assert torch.isclose(out, torch.tensor([2.0]), atol=1e-5).all()
# ② ratio=e^0.5≈1.649 > 1.2，A=+1 → 被 clip 封顶在 1.2
out = clipped_surrogate(torch.tensor([0.5]), torch.tensor([0.0]), torch.tensor([1.0]), 0.2)
assert torch.isclose(out, torch.tensor([1.2]), atol=1e-4).all()
# ③ 同样 ratio，A=−1 → min 选更负的 r·A，不被 clip（悲观下界）
out = clipped_surrogate(torch.tensor([0.5]), torch.tensor([0.0]), torch.tensor([-1.0]), 0.2)
assert torch.isclose(out, torch.tensor([-1.6487]), atol=1e-3).all()
# ④ ratio=e^-0.5≈0.607 < 0.8，A=−1 → 惩罚到底，clip 在 −0.8
out = clipped_surrogate(torch.tensor([-0.5]), torch.tensor([0.0]), torch.tensor([-1.0]), 0.2)
assert torch.isclose(out, torch.tensor([-0.8]), atol=1e-4).all()
# ⑤ 向量化
out = clipped_surrogate(torch.tensor([0.0, 0.5, 0.5, -0.5]), torch.zeros(4),
                        torch.tensor([2.0, 1.0, -1.0, -1.0]), 0.2)
assert out.shape == (4,)
assert torch.allclose(out, torch.tensor([2.0, 1.2, -1.6487, -0.8]), atol=1e-3)
print("✅ 练习 1 通过")

## ✏️ 练习 2：实现 GAE

实现 `gae(rewards, values, gamma, lam)`：

- `rewards`：长度 $T$ 的张量；`values`：长度 $T+1$（最后一个是 bootstrap 值 $V(s_T)$，episode 终止时传 0）；
- 返回长度 $T$ 的 advantage 张量。

公式（讲义 §5）：$\delta_t = r_t + \gamma V_{t+1} - V_t$，反向递推 $\hat A_t = \delta_t + \gamma\lambda \hat A_{t+1}$（$\hat A_T = 0$）。

提示：从 $t=T-1$ 倒着循环，10 行以内。自测包含一个**手算的 3 步序列**和两个极端（$\lambda=0$ → 纯 TD；$\gamma=\lambda=1$ → 纯 MC）。

In [ ]:
def gae(rewards, values, gamma, lam):
    # TODO: delta_t = rewards[t] + gamma*values[t+1] - values[t]
    # TODO: 从后往前累积 A_t = delta_t + gamma*lam*A_{t+1}
    # 返回与 rewards 同长度的张量
    raise NotImplementedError

In [ ]:
# ---- 自测：练习 2 ----
r3 = torch.tensor([1.0, 0.0, 2.0])
v3 = torch.tensor([0.5, 1.0, 0.2, 0.0])
# 手算（γ=0.9, λ=0.8, γλ=0.72）：
#   δ2 = 2 + 0       − 0.2 = 1.8          → A2 = 1.8
#   δ1 = 0 + 0.9·0.2 − 1.0 = −0.82        → A1 = −0.82 + 0.72·1.8  = 0.476
#   δ0 = 1 + 0.9·1.0 − 0.5 = 1.4          → A0 = 1.4  + 0.72·0.476 = 1.74272
out = gae(r3, v3, gamma=0.9, lam=0.8)
assert out.shape == (3,)
assert torch.allclose(out, torch.tensor([1.74272, 0.476, 1.8]), atol=1e-5)
# 边界 ①：λ=0 退化为单步 TD 残差 δ
out0 = gae(r3, v3, gamma=0.9, lam=0.0)
assert torch.allclose(out0, torch.tensor([1.4, -0.82, 1.8]), atol=1e-5)
# 边界 ②：γ=λ=1 退化为 Monte Carlo：A_t = Σ_{l≥t} r_l − V_t
out1 = gae(r3, v3, gamma=1.0, lam=1.0)
assert torch.allclose(out1, torch.tensor([2.5, 1.0, 1.8]), atol=1e-5)
print("✅ 练习 2 通过")

## ✏️ 练习 3：per-token KL 惩罚 reward

实现 `kl_penalty_reward(r, logp, logp_ref, beta)`（讲义 §6 的公式）：

- `r`：RM 给**整条回答**的标量分数；`logp` / `logp_ref`：policy / reference 在每个生成 token 上的 log 概率（长度 $T$）；
- 返回长度 $T$ 的 per-token reward：每个位置 $-\beta(\log\pi_t - \log\pi_{\mathrm{ref},t})$，**RM 分数只加在最后一个 token 上**。

提示：2–3 行。注意符号方向：policy 比 ref 更"自信"（$\log\pi > \log\pi_\mathrm{ref}$）的 token 应得到**负**奖励（被罚）。不要原地修改输入。

In [ ]:
def kl_penalty_reward(r, logp, logp_ref, beta):
    # TODO: out = -beta * (logp - logp_ref)
    # TODO: 最后一个位置加上序列级 RM 分数 r（注意 clone，别改输入）
    raise NotImplementedError

In [ ]:
# ---- 自测：练习 3 ----
out = kl_penalty_reward(2.0, torch.tensor([-1.0, -0.5]), torch.tensor([-1.2, -0.9]), beta=0.1)
assert out.shape == (2,)
# kl 项：(-1.0+1.2)=0.2 → −0.02；(-0.5+0.9)=0.4 → −0.04；末位 +2.0 → 1.96
assert torch.allclose(out, torch.tensor([-0.02, 1.96]), atol=1e-6)
# 符号方向：policy 偏离 ref 越自信，非末位 reward 越负
assert out[0] < 0
# π = π_ref 时：KL 项全为 0，只剩末位的 RM 分数
out2 = kl_penalty_reward(1.5, torch.tensor([-1.0, -2.0, -0.3]), torch.tensor([-1.0, -2.0, -0.3]), beta=0.2)
assert torch.allclose(out2, torch.tensor([0.0, 0.0, 1.5]), atol=1e-6)
# β=0 时退化为"只有末位 RM 分数"
out3 = kl_penalty_reward(0.7, torch.tensor([-0.1, -0.2]), torch.tensor([-2.0, -2.0]), beta=0.0)
assert torch.allclose(out3, torch.tensor([0.0, 0.7]), atol=1e-6)
print("✅ 练习 3 通过")

## 📖 参考答案

先自己做，再对照。

In [ ]:
# 参考答案 1（先自己做，再对照）
def clipped_surrogate(logp_new, logp_old, adv, eps):
    ratio = torch.exp(logp_new - logp_old)
    return torch.min(ratio * adv, torch.clamp(ratio, 1 - eps, 1 + eps) * adv)

In [ ]:
# 参考答案 2（先自己做，再对照）
def gae(rewards, values, gamma, lam):
    T = rewards.shape[0]
    adv = torch.zeros(T)
    running = 0.0
    for t in range(T - 1, -1, -1):
        delta = rewards[t] + gamma * values[t + 1] - values[t]
        running = delta + gamma * lam * running
        adv[t] = running
    return adv

In [ ]:
# 参考答案 3（先自己做，再对照）
def kl_penalty_reward(r, logp, logp_ref, beta):
    out = -beta * (logp - logp_ref)
    out = out.clone()
    out[-1] = out[-1] + r
    return out

## 小结

你已经在一个 20 维玩具上复现了 RLHF 第三阶段的全部机制链条：

- **缺陷 RM**（BT loss 学到被污染的偏好）→ proxy 与 true reward 的裂缝；
- **REINFORCE → baseline → advantage**：方差是 policy gradient 的头号敌人；
- **PPO**：importance ratio + clipped surrogate 让同一批昂贵的 rollout 可以安全地多 epoch 重用 [Schulman 2017]；
- **KL 惩罚消融**：β=0 → 收敛到 hack 回答、KL 飙升；β 适中 → 收敛到真好回答 —— "KL 是防 reward hacking 的缰绳"不再是口号 [Ouyang 2022]；
- 三道练习覆盖了 LLM-PPO 实现的三块核心积木：clip、GAE、per-token KL reward [Zheng 2023]。

**留一个评测者的问题**：本实验里我们有 gold reward 才看穿了 hack；真实世界没有 gold，你会用什么信号替代它？（提示：held-out RM、人评抽检、KL 预算 —— 讲义 §8。）

**下一站 → 模块 04 · DPO 家族**：既然 PPO 这么贵又这么难调（四个模型、十几个超参），能不能把"reward − β·KL"这个目标**解析地**变成一个监督学习损失，跳过 RL 全程？DPO 的答案令人惊讶地优雅。

---
## 🎯 真实数据胶囊题：真实奖励上的 PPO 裁剪目标

PPO 用裁剪的 surrogate objective 防止策略一步走太远。用真实红酒质量当奖励信号、算优势，实现裁剪目标，验证它在 ratio 过大时被截断（这正是 PPO 稳定的来源）。

> 本模块新增的**真实数据**练习：自包含、用真实公开数据把本章技术跑一遍。先做 TODO，`assert` 全过即通关，文末有参考答案。

In [ ]:
import os, json, urllib.request, re
import numpy as np
CACHE=os.path.expanduser("~/.post_training_data"); os.makedirs(CACHE,exist_ok=True)
def _fetch(url,fn):
    p=os.path.join(CACHE,fn)
    if not os.path.exists(p): urllib.request.urlretrieve(url,p)
    return p
def gsm8k(n=200):
    p=_fetch("https://raw.githubusercontent.com/openai/grade-school-math/master/grade_school_math/data/test.jsonl","gsm8k_test.jsonl")
    rows=[json.loads(l) for l in open(p).read().splitlines()[:n]]
    return rows
def winequality():
    import pandas as pd
    p=_fetch("https://archive.ics.uci.edu/ml/machine-learning-databases/wine-quality/winequality-red.csv","winequality-red.csv")
    return pd.read_csv(p, sep=";")

df = winequality()
reward = ((df["quality"]-df["quality"].mean())/df["quality"].std()).to_numpy()  # 真实标准化奖励
adv = reward - reward.mean()    # 简单优势 = 奖励 - 基线
print(f"真实奖励(标准化质量) 范围 [{reward.min():.2f},{reward.max():.2f}], 均值≈0")

**练习**：实现 `ppo_clip_objective(ratios, advantages, eps)` = `mean(min(r·A, clip(r,1-eps,1+eps)·A))`。验证大 ratio + 正优势时被裁剪。

In [ ]:
def ppo_clip_objective(ratios, advantages, eps=0.2):
    # TODO: min(r*A, clip(r,1-eps,1+eps)*A) 的均值
    raise NotImplementedError


In [ ]:
# 自测
A = adv[:100]
# ratio=1 时目标 = mean(A)
assert abs(ppo_clip_objective(np.ones(100), A) - A.mean()) < 1e-9
# 巨大 ratio + 正优势：被裁剪到 (1+eps)*A，小于不裁剪
r_big=np.full(100, 5.0)
clipped=ppo_clip_objective(r_big, A, 0.2)
unclipped=(r_big*A).mean()
pos=A>0
assert clipped < unclipped, "正优势处大 ratio 应被裁剪，目标更小"
print(f"PPO 裁剪 ✓  裁剪后目标 {clipped:.3f} < 未裁剪 {unclipped:.3f}")


### 📖 参考答案

In [ ]:
def ppo_clip_objective(ratios, advantages, eps=0.2):
    unclip = ratios*advantages
    clip = np.clip(ratios, 1-eps, 1+eps)*advantages
    return float(np.minimum(unclip, clip).mean())
print("✓ min + clip 让策略更新有'信任域'，是 PPO 稳定的核心")